# 01 — Generate AUDIO-grounded Tool-Calling Data

**Run on a GPU Modal notebook** (Mimi encoding needs GPU; TTS is CPU/network).

The user's question goes into the **audio** rows of the code tensor, so Moshi
learns to emit `<|tool_call|>` when it *hears* a request. Phase-1b also trains
the listening/idle frames to PAD so the model stays quiet otherwise.

Per example, `codes[17, T]`:

| rows | stream | content |
|------|--------|---------|
| `0` | text monologue | PAD while listening/idle, `<|tool_call|>…` at the request, then spoken reply |
| `1:9` | Moshi audio | silence |
| `9:17` | **user audio** | edge-tts speech of the question, Mimi-encoded |

Pipeline: `gen_tool_data.examples()` → edge-tts (15 voices, disk-cached) → Mimi
encode → `codes[17,T]` → push as an HF **dataset** (`abrarfahim/moshi-tool-audio`,
playable `audio` column). Notebook 02 loads it and trains.

Run top-to-bottom. TTS is cached on disk by (voice, text); rendering always runs
fresh, so logic changes never train on stale data.

In [ ]:
# Install into the EXACT python running this kernel.
# datasets<3.0 encodes audio via soundfile (no torchcodec/ffmpeg) -> playable Audio column.
import sys
!{sys.executable} -m pip install edge-tts sphn nest_asyncio sentencepiece huggingface_hub einops numpy tqdm "datasets<3.0" soundfile -q
import importlib
for m in ["edge_tts", "sphn", "nest_asyncio", "sentencepiece", "einops",
          "huggingface_hub", "tqdm", "datasets", "soundfile"]:
    importlib.import_module(m)
import datasets
print(f"datasets {datasets.__version__}  ✓ (must be <3.0)")
print("If datasets was already imported as 3.x in this kernel, RESTART the kernel now.")

In [ ]:
import sys, gc, os, shutil, subprocess, asyncio, tempfile
from pathlib import Path
import numpy as np
import torch

# ── Clone repo ────────────────────────────────────────────────────────────────
REPO = Path("/repo")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone",
    "https://github.com/syedfahimabrar/moshi-D-gu.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "moshi"))
sys.path.insert(0, str(REPO / "notebooks"))

import sentencepiece as spm
from moshi.models import loaders
import gen_tool_data as G

# ── Config ────────────────────────────────────────────────────────────────────
HF_PATCHED_REPO = "abrarfahim/moshi-tool-patched"   # tokenizer source
HF_DATA_REPO    = "abrarfahim/moshi-tool-audio"     # where we cache the dataset
HF_TOKEN        = os.environ.get("HF_TOKEN", None)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR     = 24000          # Mimi sample rate
FRAME  = 1920          # samples per 12.5 Hz frame (24000 / 12.5)

# edge-tts voices — variety of accents/genders so triggering generalises
VOICES = [
    "en-US-AriaNeural", "en-US-GuyNeural", "en-US-JennyNeural",
    "en-US-MichelleNeural", "en-US-ChristopherNeural", "en-US-EricNeural",
    "en-GB-SoniaNeural", "en-GB-RyanNeural", "en-GB-LibbyNeural",
    "en-AU-NatashaNeural", "en-AU-WilliamNeural",
    "en-IN-NeerjaNeural", "en-IN-PrabhatNeural",
    "en-CA-ClaraNeural", "en-CA-LiamNeural",
]
print(f"Device: {DEVICE} | voices: {len(VOICES)}")

In [ ]:
from huggingface_hub import hf_hub_download

# Tokenizer (patched, 32004 tokens)
tok_path = hf_hub_download(HF_PATCHED_REPO, "tokenizer_spm_32k_3.model", token=HF_TOKEN)
tok = spm.SentencePieceProcessor(); tok.Load(tok_path)
assert tok.get_piece_size() == 32004, tok.get_piece_size()
assert tok.id_to_piece(G.TOOL_CALL_ID) == "<|tool_call|>"
print(f"Tokenizer ✓ ({tok.get_piece_size()} tokens)")

# Mimi audio codec (from PersonaPlex), 8 codebooks
mimi_path = hf_hub_download(loaders.DEFAULT_REPO, loaders.MIMI_NAME, token=HF_TOKEN)
mimi = loaders.get_mimi(mimi_path, device=DEVICE)
mimi.eval()

# Verify N*FRAME samples -> N frames, and 8 codebooks
with torch.no_grad():
    probe = mimi.encode(torch.zeros(1, 1, FRAME * 10, device=DEVICE))
print(f"Mimi ✓  encode(10 frames) -> {tuple(probe.shape)}  (expect [1, 8, 10])")
assert probe.shape[1] == 8

In [ ]:
import edge_tts, sphn, nest_asyncio, hashlib, soundfile as sf
nest_asyncio.apply()   # allow asyncio.run inside the Jupyter event loop

def _resample(wav, src, dst):
    if src == dst:
        return wav.astype("float32")
    n = int(round(len(wav) * dst / src))
    x  = np.linspace(0, 1, len(wav), endpoint=False)
    xn = np.linspace(0, 1, n, endpoint=False)
    return np.interp(xn, x, wav).astype("float32")

# TTS clips are cached on DISK by (voice, text) so they survive kernel restarts.
# Only the audio is cached — rendering always runs fresh, so changes to the
# masking/structure logic can never train on stale data.
TTS_DIR = Path("/tmp/tts_cache"); TTS_DIR.mkdir(exist_ok=True)
_tts_mem = {}

def _tts_path(text, voice):
    h = hashlib.md5(f"{voice}|{text}".encode()).hexdigest()
    return TTS_DIR / f"{h}.wav"

async def _fetch_one(text, voice, sem, retries=3):
    if _tts_path(text, voice).exists():
        return
    async with sem:
        for attempt in range(retries):
            try:
                with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as fh:
                    mp3 = fh.name
                await edge_tts.Communicate(text, voice).save(mp3)
                wav, sr = sphn.read(mp3); os.unlink(mp3)
                if wav.ndim > 1:
                    wav = wav.mean(axis=0)
                sf.write(str(_tts_path(text, voice)), _resample(wav, sr, SR), SR)
                return
            except Exception:
                if attempt == retries - 1:
                    raise
                await asyncio.sleep(1.0)

async def _prefetch(pairs, concurrency=16):
    sem = asyncio.Semaphore(concurrency)
    todo = [(t, v) for (t, v) in pairs if not _tts_path(t, v).exists()]
    for i in range(0, len(todo), 200):
        await asyncio.gather(*[_fetch_one(t, v, sem) for (t, v) in todo[i:i + 200]])
        print(f"  TTS {min(i+200, len(todo))}/{len(todo)}")

def prefetch_tts(pairs, concurrency=16):
    asyncio.run(_prefetch(list(pairs), concurrency))

def tts_wav(text, voice):
    key = (text, voice)
    if key not in _tts_mem:
        p = _tts_path(text, voice)
        if not p.exists():
            asyncio.run(_fetch_one(text, voice, asyncio.Semaphore(1)))
        _tts_mem[key], _ = sf.read(str(p), dtype="float32")
    return _tts_mem[key]

# smoke test
prefetch_tts([("what time is it", VOICES[0])])
print(f"TTS ✓  ({len(tts_wav('what time is it', VOICES[0]))} samples)")

In [ ]:
PAD_ID = G.PAD_ID   # bring the special-token id into this namespace

def _pad_to_frames(wav):
    n = len(wav)
    fr = int(np.ceil(n / FRAME)) if n > 0 else 0
    out = np.zeros(fr * FRAME, dtype="float32")
    out[:n] = wav
    return out, fr

def _silence(frames):
    return np.zeros(frames * FRAME, dtype="float32")

@torch.no_grad()
def _encode(wav):
    t = torch.from_numpy(wav).to(DEVICE)[None, None]
    return mimi.encode(t)[0].cpu()

_SIL = {"codes": None}
def _silence_codes(T):
    c = _SIL["codes"]
    if c is None or c.shape[1] < T:
        _SIL["codes"] = _encode(_silence(max(T + 8, 1200)))
    return _SIL["codes"][:, :T].clone()

def _fit(codes, T):
    Ta = codes.shape[1]
    if Ta < T:
        codes = torch.cat([codes, codes[:, -1:].repeat(1, T - Ta)], dim=1)
    elif Ta > T:
        codes = codes[:, :T]
    return codes

def render(ex, voice):
    """Abstract Example -> {'codes':[17,T], 'mask':[T], ...}.

    CALIBRATION (Phase-1b): listening / gap / idle frames are now TRAINED
    (mask=1) to predict PAD, so the model learns to keep <|tool_call|> down
    except at the actual request. Only the call body and spoken reply differ.
    """
    user_parts, text, mask = [], [], []
    queries, replies = [], []

    def add_idle(n):                       # trained silence (mask=1 -> PAD)
        text.extend([PAD_ID] * n)
        mask.extend([1] * n)
        user_parts.append(_silence(n))

    add_idle(int(np.random.randint(5, 15)))           # leading idle

    for turn in ex.turns:
        if turn.query is None:                         # pure-silence turn
            add_idle(int(np.random.randint(20, 45)))
            continue
        qwav, fq = _pad_to_frames(tts_wav(turn.query, voice))
        gap = int(np.random.randint(2, 6))
        emit_t, emit_m = G.render_emit(turn, tok)
        E = len(emit_t)
        # listening + gap: stay silent (PAD), trained -> suppresses tool token
        text.extend([PAD_ID] * (fq + gap)); mask.extend([1] * (fq + gap))
        user_parts.append(np.concatenate([qwav, _silence(gap)]))
        # emit: call (mask1) + result block (mask0, injected) + reply (mask1)
        text.extend(emit_t); mask.extend(emit_m)
        user_parts.append(_silence(E))
        queries.append(turn.query); replies.append(turn.reply)

    add_idle(int(np.random.randint(5, 15)))           # trailing idle

    user_wav = np.concatenate(user_parts)
    T = len(text)
    codes = torch.zeros(17, T, dtype=torch.long)
    codes[0]    = torch.tensor(text, dtype=torch.long)
    codes[1:9]  = _silence_codes(T)
    codes[9:17] = _fit(_encode(user_wav), T)
    return {
        "type":  ex.type,
        "query": " | ".join(queries),
        "reply": " | ".join(replies),
        "audio_array": user_wav.astype("float32"),
        "codes": codes.to(torch.int32).tolist(),
        "mask":  [int(m) for m in mask],
        "voice": voice,
    }

print("render helpers ready ✓ (Phase-1b: listening/idle trained to suppress tool token)")

In [ ]:
from tqdm.auto import tqdm

exs = G.examples(seed=42)

# 1) Prefetch all unique (query, voice) TTS clips concurrently (cached on disk)
pairs = set()
for i, ex in enumerate(exs):
    v = VOICES[i % len(VOICES)]
    for turn in ex.turns:
        if turn.query:
            pairs.add((turn.query, v))
print(f"Ensuring {len(pairs)} unique TTS clips (concurrent) ...")
prefetch_tts(pairs, concurrency=16)

# 2) Render fresh every run (Mimi encode on GPU; cheap vs TTS) — never stale
np.random.seed(42)
records = [render(ex, VOICES[i % len(VOICES)]) for i, ex in enumerate(tqdm(exs, desc="render"))]

from collections import Counter
print(f"\nBuilt {len(records)} records")
print("Breakdown:", dict(Counter(r["type"] for r in records)))
print(f"Avg frames: {sum(len(r['mask']) for r in records)/len(records):.0f}  "
      f"Max frames: {max(len(r['mask']) for r in records)}")
# sanity: fraction of trained frames that are the tool call (should be small)
import itertools
tot = sum(len(r['mask']) for r in records)
calls = sum(r['codes'][0].count(G.TOOL_CALL_ID) for r in records)
print(f"Tool-call frames: {calls} / {tot} total ({100*calls/tot:.2f}%)")

In [ ]:
# Inspect the data as a dataframe + play audio inline (run before the push cell)
import pandas as pd
from IPython.display import Audio, display

df = pd.DataFrame([{
    "type":    r["type"],
    "query":   r["query"],
    "reply":   r["reply"],
    "voice":   r["voice"],
    "frames":  len(r["mask"]),
    "trained": int(sum(r["mask"])),
    "calls":   r["codes"][0].count(G.TOOL_CALL_ID),
} for r in records])

print("Counts by type:")
display(df.groupby("type").size().rename("count").to_frame())
print("\nRandom sample:")
display(df.sample(min(12, len(df)))[["type", "query", "reply", "voice", "frames", "calls"]])

# Play one example per type (uses the in-memory waveform before the push cell pops it)
print("\nListen — one per type:")
for t in sorted(set(r["type"] for r in records)):
    r = next(x for x in records if x["type"] == t)
    arr = r["audio"]["array"] if isinstance(r.get("audio"), dict) else r["audio_array"]
    print(f"[{t}]  query={r['query']!r}")
    display(Audio(np.asarray(arr, dtype='float32'), rate=SR))

In [ ]:
# Build a proper HF dataset with a PLAYABLE audio column, push, and add a card.
# Requires datasets<3.0 (encodes audio arrays via soundfile, no torchcodec).
from datasets import Dataset, Features, Value, Sequence, Audio
from huggingface_hub import HfApi

# attach the user waveform as an Audio column (array dict; encoded by soundfile)
for r in records:
    if "audio" not in r:
        r["audio"] = {"array": np.asarray(r.pop("audio_array"), dtype="float32"),
                      "sampling_rate": SR}

features = Features({
    "type":  Value("string"),
    "query": Value("string"),
    "reply": Value("string"),
    "voice": Value("string"),
    "audio": Audio(sampling_rate=SR),                   # playable in the HF viewer
    "codes": Sequence(Sequence(Value("int32"))),       # 17 x T training cache
    "mask":  Sequence(Value("int8")),
})

ds = Dataset.from_list(records, features=features)
print(ds)
ds.push_to_hub(HF_DATA_REPO, private=True, token=HF_TOKEN)

# ── Dataset card (README.md) ──────────────────────────────────────────────────
from collections import Counter
counts = Counter(r["type"] for r in records)
rows = "\n".join(f"| `{k}` | {counts[k]} |" for k in sorted(counts))
card = f"""---
license: mit
task_categories:
- automatic-speech-recognition
language:
- en
tags:
- moshi
- personaplex
- tool-calling
- speech
size_categories:
- 1K<n<10K
---

# Moshi Tool-Calling — Audio-Grounded Dataset

Audio-grounded data teaching **Moshi / PersonaPlex** to emit tool-call special
tokens in its inner monologue **when it hears** a request — and to stay quiet
otherwise (listening/idle frames are trained to PAD).

Each row is a code tensor `codes[17, T]` at 12.5 Hz:

| rows | stream | content |
|------|--------|---------|
| `0` | text monologue | PAD while listening/idle, `<|tool_call|>…<|tool_end|>` at the request, then spoken reply |
| `1:9` | Moshi audio | silence |
| `9:17` | **user audio** | the spoken question (edge-tts), Mimi-encoded |

`mask`=1 marks frames the model must produce (PAD suppression + call + reply);
the injected `<|tool_result|>` block is context (mask 0).

## Columns
`type`, `query`, `reply`, `voice`, `audio` (24 kHz, playable), `codes` (17×T int), `mask` (T).

## Special tokens
`<|tool_call|>`=32000, `<|tool_end|>`=32001, `<|tool_result|>`=32002, `<|tool_result_end|>`=32003

## Tools
`get_time`, `get_weather <city>`.

## Composition ({len(records)} examples)

| type | count |
|------|------|
{rows}

Built by `notebooks/01_generate_data.ipynb` in
[moshi-D-gu](https://github.com/syedfahimabrar/moshi-D-gu).
"""

HfApi(token=HF_TOKEN).upload_file(
    path_or_fileobj=card.encode(), path_in_repo="README.md",
    repo_id=HF_DATA_REPO, repo_type="dataset", token=HF_TOKEN)

print(f"\nPushed + card → https://huggingface.co/datasets/{HF_DATA_REPO}")
print("Next: run notebook 02 (it loads this dataset and trains).")